# 結合と連結

In [1]:
import polars as pl

## 結合

### 単一の結合

In [2]:
df_left = pl.DataFrame({
    'key': ['A', 'B', 'C', 'D'],
    'value': [1, 2, 3, 4]
})

df_right = pl.DataFrame({
    'key': ['B', 'C', 'D', 'E'],
    'value': [5, 6, 7, 8]
})

In [3]:
df_left.join(df_right, on='key', how='inner')

key,value,value_right
str,i64,i64
"""B""",2,5
"""C""",3,6
"""D""",4,7


In [4]:
df_left.join(df_right, on='key', how='full', suffix='_ohter')

key,value,key_ohter,value_ohter
str,i64,str,i64
"""B""",2,"""B""",5
"""C""",3,"""C""",6
"""D""",4,"""D""",7
null,null,"""E""",8
"""A""",1,null,null


outer joinを指定するときは、今のpolarsだと`how='outer'`ではなくて`how='full'`と指定するのが推奨とのこと。  
`how=outer`だと何が良くなかったんだろう？

In [5]:
df_left.join(df_right, on='key', how='left')

key,value,value_right
str,i64,i64
"""A""",1,null
"""B""",2,5
"""C""",3,6
"""D""",4,7


In [6]:
df_left.join(df_right, how='cross')

key,value,key_right,value_right
str,i64,str,i64
"""A""",1,"""B""",5
"""A""",1,"""C""",6
"""A""",1,"""D""",7
"""A""",1,"""E""",8
"""B""",2,"""B""",5
…,…,…,…
"""C""",3,"""E""",8
"""D""",4,"""B""",5
"""D""",4,"""C""",6


`how=cross`ってあるんだ

In [7]:
df_left.join(df_right, on='key', how='semi')

key,value
str,i64
"""B""",2
"""C""",3
"""D""",4


直感的だと`how='right'`な感じか。`how='semi'`によって、結合する側のキーに一致するテーブルが作れる

In [8]:
df_left.join(df_right, on='key', how='anti')

key,value
str,i64
"""A""",1


`how='semi'`の逆が`how='anti'`。結合する側のテーブルに一致しない結合される側のテーブルのみを抽出する。差集合みたいなイメージ

### 複数の結合

In [9]:
df_left = pl.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'Dave'],
    'city': ['NY', 'LA', 'NY', 'SF'],
    'age': [25, 30, 35, 40]
})

df_right = pl.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'Dave'],
    'city': ['NY', 'LA', 'NY', 'Chicago'],
    'department': ['Finance', 'Marketing', 'Engineering', 'Operations']
})

df_left.join(df_right, on=['name', 'city'], how='inner')

name,city,age,department
str,str,i64,str
"""Alice""","""NY""",25,"""Finance"""
"""Bob""","""LA""",30,"""Marketing"""
"""Charlie""","""NY""",35,"""Engineering"""


### 検証

In [10]:
df_employees = pl.DataFrame({
    'employee_id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Charlie', 'Dave'],
    'department_id': [10, 10, 30, 10]
})

df_departments = pl.DataFrame({
    'department_id': [10, 20, 30],
    'department_name': ['Information Technology', 'Finance', 'Human Resources']
})

df_employees.join(
    df_departments,
    on='department_id',
    how='left',
    validate='m:1'
)

employee_id,name,department_id,department_name
i64,str,i64,str
1,"""Alice""",10,"""Information Technology"""
2,"""Bob""",10,"""Information Technology"""
3,"""Charlie""",30,"""Human Resources"""
4,"""Dave""",10,"""Information Technology"""


複数のカラムを指定した結合を実施する場合は、`validate=m:1`と言ったように`m:m / 1:m / 1:1`といった組み合わせを指定する必要がある

In [11]:
df_departments = pl.DataFrame({
    'department_id': [10, 20, 10],
    'department_name': ['Information Technology', 'Finance', 'Human Resources']
})

df_employees.join(
    df_departments,
    on='department_id',
    how='left',
    validate='m:m'
)

employee_id,name,department_id,department_name
i64,str,i64,str
1,"""Alice""",10,"""Information Technology"""
1,"""Alice""",10,"""Human Resources"""
2,"""Bob""",10,"""Information Technology"""
2,"""Bob""",10,"""Human Resources"""
3,"""Charlie""",30,null
4,"""Dave""",10,"""Information Technology"""
4,"""Dave""",10,"""Human Resources"""


### 不正確な結合

In [12]:
df_left = pl.DataFrame({
    'int_id': [5, 10],
    'value': ['1', '2']
})

df_right = pl.DataFrame({
    'int_id': [4, 7, 12],
    'value': [1, 2, 3]
})

df_left.join_asof(df_right, on='int_id', tolerance=3)

int_id,value,value_right
i64,str,i64
5,"""1""",1
10,"""2""",2


`tolerance`パラメータで指定した数値は、等しいと判定するときに指標となる差分（=絶対値）を指定している。  
この場合は、差分が3以内であれば不正確でも結合するという意味

In [13]:
df_right = df_right.rename({'int_id': 'int_id_right'})

df_left.join_asof(
    df_right,
    left_on='int_id',
    right_on='int_id_right'
)

int_id,value,int_id_right,value_right
i64,str,i64,i64
5,"""1""",4,1
10,"""2""",7,2


In [14]:
df_left.join_asof(
    df_right,
    left_on='int_id',
    right_on='int_id_right',
    tolerance=3,
    strategy='backward'
)

int_id,value,int_id_right,value_right
i64,str,i64,i64
5,"""1""",4,1
10,"""2""",7,2


`strategy='backward'`では、左側のテーブルの値と等しいorそれより小さい値と結合する。  
デフォルト値はこれ

In [15]:
df_left.join_asof(
    df_right,
    left_on='int_id',
    right_on='int_id_right',
    tolerance=3,
    strategy='forward'
)

int_id,value,int_id_right,value_right
i64,str,i64,i64
5,"""1""",7,2
10,"""2""",12,3


`strategy='forward'`は、左側のテーブルの値と等しいorそれより大きい値と結合する

In [16]:
df_left.join_asof(
    df_right,
    left_on='int_id',
    right_on='int_id_right',
    tolerance=3,
    strategy='nearest'
)

int_id,value,int_id_right,value_right
i64,str,i64,i64
5,"""1""",4,1
10,"""2""",12,3


`strategy=nearest`では、左側と右側で最も近い行を対象に結合する

## 連結

In [17]:
df1 = pl.DataFrame({
    'id': [1, 2, 3],
    'value': ['a', 'b', 'c']
})
df2 = pl.DataFrame({
    'id': [4, 5],
    'value': ['d', 'e']
})

pl.concat([df1, df2], how='vertical')

id,value
i64,str
1,"""a"""
2,"""b"""
3,"""c"""
4,"""d"""
5,"""e"""


In [18]:
df1 = pl.DataFrame({
    'id': [1, 2, 3],
    'value': ['a', 'b', 'c']
})
df2 = pl.DataFrame({
    'value2': ['x', 'y']
})

pl.concat([df1, df2], how='horizontal')

id,value,value2
i64,str,str
1,"""a""","""x"""
2,"""b""","""y"""
3,"""c""",null


In [19]:
df1 = pl.DataFrame({
    'id': [1, 2, 3],
    'value': ['a', 'b', 'c']
})
df2 = pl.DataFrame({
    'value': ['d', 'e'],
    'value2': ['x', 'y']
})

pl.concat([df1, df2], how='diagonal')

id,value,value2
i64,str,str
1,"""a""",null
2,"""b""",null
3,"""c""",null
null,"""d""","""x"""
null,"""e""","""y"""


対角線上に連結する方法。共通のカラム名があれば、そっちの垂直方向に結合して、それ以外が水平方向に伸びる

In [20]:
df1 = pl.DataFrame({
    'id': [1, 2, 3],
    'value': ['a', 'b', 'c']
})
df2 = pl.DataFrame({
    'value': ['a', 'b', 'c'],
    'value2': ['x', 'y', 'z']
})

pl.concat([df1, df2], how='align')

id,value,value2
i64,str,str
1,"""a""","""x"""
2,"""b""","""y"""
3,"""c""","""z"""


`how=align`では、キーとなる要素を見つけて、それに合わせた結合が実行される

In [21]:
df1 = pl.DataFrame({
    'id': [1, 2, 3],
    'value': ['a', 'b', 'c']
})
df2 = pl.DataFrame({
    'id': [4.0, 5.0],
    'value': [1, 2]
})

pl.concat([df1, df2], how='vertical_relaxed')

id,value
f64,str
1.0,"""a"""
2.0,"""b"""
3.0,"""c"""
4.0,"""1"""
5.0,"""2"""


`how='vertical'`と指定すると、int型とfloat型とで型が不一致であるエラーが出る。  
しかし、`how=vertical_relaxed`によって型の厳密な結合を無視できる。  
relaxって命名がなんかおしゃれ

In [22]:
df1 = pl.DataFrame({
    'id': [1, 2, 2],
    'value': ['a', 'c', 'b']
})
df2 = pl.DataFrame({
    'id': [2, 2],
    'value': ['x', 'y']
})

pl.align_frames(df1, df2, on='id')

[shape: (5, 2)
 ┌─────┬───────┐
 │ id  ┆ value │
 │ --- ┆ ---   │
 │ i64 ┆ str   │
 ╞═════╪═══════╡
 │ 1   ┆ a     │
 │ 2   ┆ c     │
 │ 2   ┆ c     │
 │ 2   ┆ b     │
 │ 2   ┆ b     │
 └─────┴───────┘,
 shape: (5, 2)
 ┌─────┬───────┐
 │ id  ┆ value │
 │ --- ┆ ---   │
 │ i64 ┆ str   │
 ╞═════╪═══════╡
 │ 1   ┆ null  │
 │ 2   ┆ x     │
 │ 2   ┆ y     │
 │ 2   ┆ x     │
 │ 2   ┆ y     │
 └─────┴───────┘]

id=2の要素が複数あるので、その直積のパターン分だけテーブルが出力されてるのか

In [23]:
df1 = pl.DataFrame({
    'id': [1, 2],
    'value': ['a', 'b']
})
df2 = pl.DataFrame({
    'id': [3, 4],
    'value': ['c', 'd']
})

df1.vstack(df2)

id,value
i64,str
1,"""a"""
2,"""b"""
3,"""c"""
4,"""d"""


In [24]:
df1 = pl.DataFrame({
    'id': [1, 2],
    'value': ['a', 'b']
})
df2 = pl.DataFrame({
    'value2': ['x', 'y']
})

df1.hstack(df2)

id,value,value2
i64,str,str
1,"""a""","""x"""
2,"""b""","""y"""


In [25]:
s1 = pl.Series('a', [1, 2])
s2 = pl.Series('b', [3, 4])
s1.append(s2)

a
i64
1
2
3
4


In [26]:
df1 = pl.DataFrame({
    'id': [1, 2],
    'value': ['a', 'b']
})
df2 = pl.DataFrame({
    'id': [3, 4],
    'value': ['c', 'd']
})

df1.extend(df2)

id,value
i64,str
1,"""a"""
2,"""b"""
3,"""c"""
4,"""d"""


Series型の結合は`.append()`で、DataFrame型の場合は`.extend()`。  
pythonのlist型に追加するときのイメージに近いかも